In [1]:
import pandas as pd 
import sqlite3

In [2]:
conn = sqlite3.connect(r"C:\Users\Dell\Downloads\inventory.db")

In [3]:
tables =pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
for table in tables['name']:
    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    df.to_csv(f"{table}.csv", index=False)
    print(f"Exported {table}.csv")

conn.close()

Exported purchases.csv
Exported purchase_prices.csv
Exported vendor_invoice.csv
Exported begin_inventory.csv
Exported end_inventory.csv


In [4]:
begin_inventory = pd.read_csv("begin_inventory.csv")
end_inventory = pd.read_csv("end_inventory.csv")
purchase_price = pd.read_csv("purchase_prices.csv")
purchase = pd.read_csv("purchases.csv")
vendor =pd.read_csv("vendor_invoice.csv")

In [5]:
vendor.head()

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,NaN
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,NaN
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,NaN
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,NaN
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,NaN


In [6]:
vendor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5543 entries, 0 to 5542
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   VendorNumber  5543 non-null   int64  
 1   VendorName    5543 non-null   object 
 2   InvoiceDate   5543 non-null   object 
 3   PONumber      5543 non-null   int64  
 4   PODate        5543 non-null   object 
 5   PayDate       5543 non-null   object 
 6   Quantity      5543 non-null   int64  
 7   Dollars       5543 non-null   float64
 8   Freight       5543 non-null   float64
 9   Approval      374 non-null    object 
dtypes: float64(2), int64(3), object(5)
memory usage: 433.2+ KB


In [7]:
vendor["PONumber"].nunique()

5543

In [8]:
vendor["VendorNumber"].nunique()

126

In [9]:
purchase.head()

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,69_MOUNTMEND_8412,69,8412,Tequila Ocho Plata Fresno,750mL,105,ALTAMAR BRANDS LLC,8124,2023-12-21,2024-01-02,2024-01-04,2024-02-16,35.71,6,214.26,1
1,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
2,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
3,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
4,76_DONCASTER_2034,76,2034,Glendalough Double Barrel,750mL,388,ATLANTIC IMPORTING COMPANY,8169,2023-12-24,2024-01-02,2024-01-09,2024-02-16,21.32,5,106.60,1


In [10]:
purchase.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2372474 entries, 0 to 2372473
Data columns (total 16 columns):
 #   Column          Dtype  
---  ------          -----  
 0   InventoryId     object 
 1   Store           int64  
 2   Brand           int64  
 3   Description     object 
 4   Size            object 
 5   VendorNumber    int64  
 6   VendorName      object 
 7   PONumber        int64  
 8   PODate          object 
 9   ReceivingDate   object 
 10  InvoiceDate     object 
 11  PayDate         object 
 12  PurchasePrice   float64
 13  Quantity        int64  
 14  Dollars         float64
 15  Classification  int64  
dtypes: float64(2), int64(6), object(8)
memory usage: 289.6+ MB


In [11]:
purchase["VendorNumber"].nunique()

126

In [12]:
po_summary = purchase.groupby("PONumber").agg({
    "Quantity" : "sum",
    "Dollars" : "sum",
    "PurchasePrice" : "mean",
    "InventoryId" : "nunique",
    "Brand" : "nunique"
}).reset_index()

po_summary.columns = [
    "PONumber",
    "Total_Quantity",
    "Total_Dollars",
    "Avg_PurchasePrice",
    "Num_Products",
    "Num_Brands"
]

In [13]:
po_summary.head()

,PONumber,Total_Quantity,Total_Dollars,Avg_PurchasePrice,Num_Products,Num_Brands
0,8106,10100,137483.78,14.336467,367,81
1,8107,24,348.72,14.530000,2,2
2,8108,8466,60281.13,7.036120,584,165
3,8109,2246,14298.09,6.675970,67,23
4,8110,8086,56493.23,7.160858,571,183


In [14]:
po_summary["PONumber"].nunique()

5543

In [15]:
po_summary.shape

(5543, 6)

In [16]:
vendor_purchase_merged = pd.merge(vendor,po_summary,on="PONumber",how="left")

In [17]:
vendor_purchase_merged.head()

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval,Total_Quantity,Total_Dollars,Avg_PurchasePrice,Num_Products,Num_Brands
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,NaN,6,214.26,35.710000,1,1
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,NaN,15,140.55,9.370000,3,2
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,NaN,5,106.60,21.320000,1,1
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,NaN,10100,137483.78,14.336467,367,81
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,NaN,1935,15527.25,8.041461,89,29


In [18]:
vendor_purchase_merged.isnull().sum()

VendorNumber            0
VendorName              0
InvoiceDate             0
PONumber                0
PODate                  0
PayDate                 0
Quantity                0
Dollars                 0
Freight                 0
Approval             5169
Total_Quantity          0
Total_Dollars           0
Avg_PurchasePrice       0
Num_Products            0
Num_Brands              0
dtype: int64

In [19]:
vendor_purchase_merged.shape

(5543, 15)

In [20]:
purchase_price.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12261 entries, 0 to 12260
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Brand           12261 non-null  int64  
 1   Description     12260 non-null  object 
 2   Price           12261 non-null  float64
 3   Size            12260 non-null  object 
 4   Volume          12260 non-null  object 
 5   Classification  12261 non-null  int64  
 6   PurchasePrice   12261 non-null  float64
 7   VendorNumber    12261 non-null  int64  
 8   VendorName      12261 non-null  object 
dtypes: float64(2), int64(3), object(4)
memory usage: 862.2+ KB


In [21]:
purchase_price["VendorNumber"].value_counts()

VendorNumber
4425     1639
9165      965
9552      960
10754     897
3252      527
         ... 
99166       1
9099        1
54          1
90059       1
5083        1
Name: count, Length: 131, dtype: int64

In [22]:
purchase_price["PurchasePrice"].nunique()

2314

In [23]:
vendor_purchase_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5543 entries, 0 to 5542
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   VendorNumber       5543 non-null   int64  
 1   VendorName         5543 non-null   object 
 2   InvoiceDate        5543 non-null   object 
 3   PONumber           5543 non-null   int64  
 4   PODate             5543 non-null   object 
 5   PayDate            5543 non-null   object 
 6   Quantity           5543 non-null   int64  
 7   Dollars            5543 non-null   float64
 8   Freight            5543 non-null   float64
 9   Approval           374 non-null    object 
 10  Total_Quantity     5543 non-null   int64  
 11  Total_Dollars      5543 non-null   float64
 12  Avg_PurchasePrice  5543 non-null   float64
 13  Num_Products       5543 non-null   int64  
 14  Num_Brands         5543 non-null   int64  
dtypes: float64(4), int64(6), object(5)
memory usage: 649.7+ KB


In [24]:
purchase.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2372474 entries, 0 to 2372473
Data columns (total 16 columns):
 #   Column          Dtype  
---  ------          -----  
 0   InventoryId     object 
 1   Store           int64  
 2   Brand           int64  
 3   Description     object 
 4   Size            object 
 5   VendorNumber    int64  
 6   VendorName      object 
 7   PONumber        int64  
 8   PODate          object 
 9   ReceivingDate   object 
 10  InvoiceDate     object 
 11  PayDate         object 
 12  PurchasePrice   float64
 13  Quantity        int64  
 14  Dollars         float64
 15  Classification  int64  
dtypes: float64(2), int64(6), object(8)
memory usage: 289.6+ MB


In [25]:
purchase["Size"].value_counts().head(20)

Size
750mL         1207700
1.75L          593298
1.5L           229841
50mL            73333
375mL           70385
3L              55911
5L              53490
Liter           47520
187mL 4 Pk      12604
4L               6138
500mL            5780
187mL            3086
250mL 4 Pk       1613
200mL 4 Pk       1397
100mL            1287
187mL 3 Pk       1267
100mL 4 Pk       1039
300mL             931
50mL 4 Pk         843
200mL             799
Name: count, dtype: int64

In [26]:
vendor_purchase_merged["Total_Quantity"].nunique()

2895

In [27]:
vendor_purchase_merged["Quantity"].nunique()

2895

In [28]:
(vendor_purchase_merged["Quantity"] == vendor_purchase_merged["Total_Quantity"]).all()
(vendor_purchase_merged["Dollars"] == vendor_purchase_merged["Total_Dollars"]).all()

np.False_

In [29]:
vendor_purchase_merged[["Quantity","Total_Quantity"]].head()

,Quantity,Total_Quantity
0,6,6
1,15,15
2,5,5
3,10100,10100
4,1935,1935


In [30]:
(vendor_purchase_merged["Quantity"] - vendor_purchase_merged["Total_Quantity"]).abs().sum()

np.int64(14494658)

In [31]:
vendor_purchase_merged[vendor_purchase_merged["Quantity"] != vendor_purchase_merged["Total_Quantity"]][
    ["PONumber","Quantity","Total_Quantity"]
].head(100)

,PONumber,Quantity,Total_Quantity
4192,12342,249,162
4193,12376,6,189
4194,12321,172,101
4195,12361,23,25328
4196,12362,25279,1719
...,...,...,...
4287,12395,78,39323
4288,12358,12,239
4289,12359,1259,143
4290,12360,5314,871


In [32]:
vendor_purchase_merged.head(20)

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval,Total_Quantity,Total_Dollars,Avg_PurchasePrice,Num_Products,Num_Brands
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,NaN,6,214.26,35.710000,1,1
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,NaN,15,140.55,9.370000,3,2
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,NaN,5,106.60,21.320000,1,1
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,NaN,10100,137483.78,14.336467,367,81
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,NaN,1935,15527.25,8.041461,89,29
5,2396,BLACK PRINCE DISTILLERY INC,2024-01-08,8191,2023-12-25,2024-02-06,23,234.83,2.30,NaN,23,234.83,10.210000,2,1
6,1128,BROWN-FORMAN CORP,2024-01-09,8150,2023-12-23,2024-02-19,4684,65403.57,1808.77,NaN,4684,65403.57,17.788554,242,53
7,1189,BULLY BOY DISTILLERS,2024-01-09,8171,2023-12-24,2024-02-04,6,132.30,5.29,NaN,6,132.30,22.050000,1,1
8,1273,CALEDONIA SPIRITS INC,2024-01-06,8172,2023-12-24,2024-02-15,5,146.80,15.53,NaN,5,146.80,29.360000,1,1
9,11567,CAMPARI AMERICA,2024-01-06,8151,2023-12-23,2024-02-20,1321,12039.71,398.71,NaN,1321,12039.71,15.917647,85,34


In [33]:
vendor_purchase_merged["InvoiceDate"]=pd.to_datetime(vendor_purchase_merged["InvoiceDate"])
vendor_purchase_merged["PayDate"]=pd.to_datetime(vendor_purchase_merged["PayDate"])

In [34]:
vendor_purchase_merged["PaymentDelay"]= (vendor_purchase_merged["PayDate"]-vendor_purchase_merged["InvoiceDate"]).dt.days

In [35]:
vendor_purchase_merged.head()

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval,Total_Quantity,Total_Dollars,Avg_PurchasePrice,Num_Products,Num_Brands,PaymentDelay
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,NaN,6,214.26,35.710000,1,1,43
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,NaN,15,140.55,9.370000,3,2,45
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,NaN,5,106.60,21.320000,1,1,38
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,NaN,10100,137483.78,14.336467,367,81,24
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,NaN,1935,15527.25,8.041461,89,29,36


In [36]:
vendor_purchase_merged["PaymentDelay"].describe()

count    5543.000000
mean       35.468519
std         5.842178
min        23.000000
25%        31.000000
50%        35.000000
75%        40.000000
max        48.000000
Name: PaymentDelay, dtype: float64

In [37]:
vendor_purchase_merged.drop("Approval",axis=1,inplace=True)

In [38]:
vendor_purchase_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5543 entries, 0 to 5542
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   VendorNumber       5543 non-null   int64         
 1   VendorName         5543 non-null   object        
 2   InvoiceDate        5543 non-null   datetime64[ns]
 3   PONumber           5543 non-null   int64         
 4   PODate             5543 non-null   object        
 5   PayDate            5543 non-null   datetime64[ns]
 6   Quantity           5543 non-null   int64         
 7   Dollars            5543 non-null   float64       
 8   Freight            5543 non-null   float64       
 9   Total_Quantity     5543 non-null   int64         
 10  Total_Dollars      5543 non-null   float64       
 11  Avg_PurchasePrice  5543 non-null   float64       
 12  Num_Products       5543 non-null   int64         
 13  Num_Brands         5543 non-null   int64         
 14  PaymentD

In [39]:
vendor_purchase_merged.loc[vendor_purchase_merged["PaymentDelay"]>40,"Risk"]=1

In [40]:
vendor_purchase_merged.head(10)

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Total_Quantity,Total_Dollars,Avg_PurchasePrice,Num_Products,Num_Brands,PaymentDelay,Risk
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,6,214.26,35.710000,1,1,43,1.0
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,15,140.55,9.370000,3,2,45,1.0
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,5,106.60,21.320000,1,1,38,NaN
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,10100,137483.78,14.336467,367,81,24,NaN
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,1935,15527.25,8.041461,89,29,36,NaN
5,2396,BLACK PRINCE DISTILLERY INC,2024-01-08,8191,2023-12-25,2024-02-06,23,234.83,2.30,23,234.83,10.210000,2,1,29,NaN
6,1128,BROWN-FORMAN CORP,2024-01-09,8150,2023-12-23,2024-02-19,4684,65403.57,1808.77,4684,65403.57,17.788554,242,53,41,1.0
7,1189,BULLY BOY DISTILLERS,2024-01-09,8171,2023-12-24,2024-02-04,6,132.30,5.29,6,132.30,22.050000,1,1,26,NaN
8,1273,CALEDONIA SPIRITS INC,2024-01-06,8172,2023-12-24,2024-02-15,5,146.80,15.53,5,146.80,29.360000,1,1,40,NaN
9,11567,CAMPARI AMERICA,2024-01-06,8151,2023-12-23,2024-02-20,1321,12039.71,398.71,1321,12039.71,15.917647,85,34,45,1.0


In [41]:
vendor_purchase_merged["Risk"]=vendor_purchase_merged["Risk"].fillna(0)

In [42]:
vendor_purchase_merged["Risk"].value_counts()

Risk
0.0    4298
1.0    1245
Name: count, dtype: int64

In [43]:
vendor_purchase_merged[vendor_purchase_merged["Risk"]==0].head()

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Total_Quantity,Total_Dollars,Avg_PurchasePrice,Num_Products,Num_Brands,PaymentDelay,Risk
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,5,106.60,21.320000,1,1,38,0.0
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,10100,137483.78,14.336467,367,81,24,0.0
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,1935,15527.25,8.041461,89,29,36,0.0
5,2396,BLACK PRINCE DISTILLERY INC,2024-01-08,8191,2023-12-25,2024-02-06,23,234.83,2.30,23,234.83,10.210000,2,1,29,0.0
7,1189,BULLY BOY DISTILLERS,2024-01-09,8171,2023-12-24,2024-02-04,6,132.30,5.29,6,132.30,22.050000,1,1,26,0.0


In [44]:
vendor_purchase_merged["Risk"]=vendor_purchase_merged["Risk"].astype(int)

In [45]:
vendor_purchase_merged[vendor_purchase_merged["Risk"]==1].head(20)

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Total_Quantity,Total_Dollars,Avg_PurchasePrice,Num_Products,Num_Brands,PaymentDelay,Risk
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,6,214.26,35.710000,1,1,43,1
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,15,140.55,9.370000,3,2,45,1
6,1128,BROWN-FORMAN CORP,2024-01-09,8150,2023-12-23,2024-02-19,4684,65403.57,1808.77,4684,65403.57,17.788554,242,53,41,1
9,11567,CAMPARI AMERICA,2024-01-06,8151,2023-12-23,2024-02-20,1321,12039.71,398.71,1321,12039.71,15.917647,85,34,45,1
11,1485,CASTLE BRANDS CORP.,2024-01-08,8152,2023-12-23,2024-02-19,320,5420.41,179.26,320,5420.41,17.256744,43,11,42,1
15,90047,CRUSH WINES,2024-01-07,8126,2023-12-21,2024-02-21,90,854.10,16.70,90,854.10,10.447500,8,6,45,1
22,3252,E & J GALLO WINERY,2024-01-09,8110,2023-12-20,2024-02-19,8086,56493.23,1300.92,8086,56493.23,7.160858,571,183,41,1
32,12546,JIM BEAM BRANDS COMPANY,2024-01-06,8142,2023-12-22,2024-02-19,11611,151890.49,3506.08,11611,151890.49,14.266473,601,158,44,1
33,4550,KLIN SPIRITS LLC,2024-01-07,8196,2023-12-25,2024-02-17,7,98.49,9.65,7,98.49,14.070000,2,1,41,1
59,7239,REMY COINTREAU USA INC,2024-01-08,8163,2023-12-23,2024-02-18,382,7515.38,348.07,382,7515.38,24.496190,42,17,41,1


In [46]:
vendor_purchase_merged.shape


(5543, 16)

In [47]:
vendor_purchase_merged.columns

Index(['VendorNumber', 'VendorName', 'InvoiceDate', 'PONumber', 'PODate',
       'PayDate', 'Quantity', 'Dollars', 'Freight', 'Total_Quantity',
       'Total_Dollars', 'Avg_PurchasePrice', 'Num_Products', 'Num_Brands',
       'PaymentDelay', 'Risk'],
      dtype='object')

In [48]:
vendor_purchase_merged[["Quantity","Dollars","Freight"]].corr()

,Quantity,Dollars,Freight
Quantity,1.000000,0.963831,0.946550
Dollars,0.963831,1.000000,0.985141
Freight,0.946550,0.985141,1.000000


In [49]:
vendor_purchase_merged[["Total_Quantity","Total_Dollars","Freight"]].corr()

,Total_Quantity,Total_Dollars,Freight
Total_Quantity,1.000000,0.963831,0.656616
Total_Dollars,0.963831,1.000000,0.670768
Freight,0.656616,0.670768,1.000000


In [50]:
vendor_purchase_merged.drop("VendorName",axis=1,inplace=True)
vendor_purchase_merged.drop("Total_Quantity",axis=1,inplace=True)
vendor_purchase_merged.drop("Total_Dollars",axis=1,inplace=True)

In [51]:
vendor_purchase_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5543 entries, 0 to 5542
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   VendorNumber       5543 non-null   int64         
 1   InvoiceDate        5543 non-null   datetime64[ns]
 2   PONumber           5543 non-null   int64         
 3   PODate             5543 non-null   object        
 4   PayDate            5543 non-null   datetime64[ns]
 5   Quantity           5543 non-null   int64         
 6   Dollars            5543 non-null   float64       
 7   Freight            5543 non-null   float64       
 8   Avg_PurchasePrice  5543 non-null   float64       
 9   Num_Products       5543 non-null   int64         
 10  Num_Brands         5543 non-null   int64         
 11  PaymentDelay       5543 non-null   int64         
 12  Risk               5543 non-null   int64         
dtypes: datetime64[ns](2), float64(3), int64(7), object(1)
memory us

In [52]:
vendor_purchase_merged.to_csv("cleaned_data.csv",index=False)